# 04. 하이퍼파라미터 튜닝 (Automatic Model Tuning)

## 이 노트북에서 배우는 것
- **탐색 범위(search space)** 정의 — 정수/연속 파라미터
- `HyperparameterTuner` 로 **베이지안 최적화** 기반 자동 튜닝 구성
- 최적화 목표 지표(`validation:mlogloss`) 와 방향(최소화) 지정
- 튜닝 결과 분석 및 **최적 모델** 선택

`01`, `02` 노트북을 먼저 실행했어야 합니다.

## 0. 환경 복원 & 튜닝 대상 Estimator 구성

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.estimator import Estimator

sess = sagemaker.Session()
role = get_execution_role()
region = sess.boto_region_name

%store -r bucket prefix train_s3 validation_s3 xgb_image_uri

xgb = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/models-hpo",
    sagemaker_session=sess,
    base_job_name="student-xgb-hpo",
)
# 튜닝하지 않는(고정) 하이퍼파라미터
xgb.set_hyperparameters(objective="multi:softprob", num_class=3, num_round=100, eval_metric="mlogloss")
print("estimator ready")

## 1. 탐색 범위 정의

각 하이퍼파라미터를 하나의 값이 아니라 **범위**로 줍니다. 튜너가 이 공간을 탐색합니다.
- `IntegerParameter(min, max)` — 정수형
- `ContinuousParameter(min, max)` — 실수형

In [ ]:
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

# TODO: 아래 범위를 완성하세요. (권장 예시는 힌트를 참고)
#  - max_depth: 정수 3 ~ 8
#  - eta(학습률): 연속 0.05 ~ 0.4
#  - 참고: https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost_hyperparameters.html
hyperparameter_ranges = {
    "max_depth": IntegerParameter(____, ____),
    "eta": ContinuousParameter(____, ____),
    "subsample": ContinuousParameter(0.6, 1.0),
    "min_child_weight": IntegerParameter(1, 6),
}
hyperparameter_ranges

## 2. Tuner 구성

`validation:mlogloss` 는 **작을수록 좋은** 지표입니다. 목표 방향을 올바르게 지정하세요.
(빌트인 XGBoost는 지표 정의가 내장되어 있어 `metric_definitions` 를 따로 주지 않아도 됩니다.)

In [ ]:
objective_metric_name = "validation:mlogloss"

tuner = HyperparameterTuner(
    estimator=xgb,
    objective_metric_name=objective_metric_name,
    # TODO: mlogloss 는 작을수록 좋습니다. 목표 방향을 지정하세요 ("Minimize" / "Maximize").
    objective_type=____,
    hyperparameter_ranges=hyperparameter_ranges,
    # TODO: 총 학습 작업 수를 지정하세요 (워크샵에서는 8 정도 권장).
    max_jobs=____,
    max_parallel_jobs=2,
    base_tuning_job_name="student-hpo",
)
print("tuner ready")

## 3. 튜닝 실행

여러 학습 작업이 순차/병렬로 실행됩니다 (10~20분 소요될 수 있습니다).

In [ ]:
from sagemaker.inputs import TrainingInput

train_input = TrainingInput(train_s3, content_type="csv")
val_input = TrainingInput(validation_s3, content_type="csv")

# TODO: 학습 때와 동일하게 train/validation 채널을 넘겨 튜닝을 시작하세요.
tuner.fit({____: train_input, ____: val_input})
tuner.wait()
print("tuning done")

## 4. 결과 분석 & 최적 모델 저장

In [ ]:
import pandas as pd

df = tuner.analytics().dataframe()
cols = [c for c in ["max_depth", "eta", "subsample", "min_child_weight",
                     "FinalObjectiveValue", "TrainingJobName"] if c in df.columns]
display(df[cols].sort_values("FinalObjectiveValue").head(10))

best_training_job = tuner.best_training_job()
best_estimator = tuner.best_estimator()
best_model_data = best_estimator.model_data
print("best training job :", best_training_job)
print("best model.tar.gz :", best_model_data)

%store best_training_job best_model_data

✅ **완료!** 최적 모델을 골랐습니다. 다음은 `05_deployment.ipynb` — 실시간 엔드포인트로 배포합니다.